# C2 · Extracción por apertura

**Spec:** [`docs/spec_C2_codex_aperture_extraction.md`](../docs/spec_C2_codex_aperture_extraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs12b_realigned`

Extrae el espectro del compañero por apertura, con controles al mismo radio.

| | |
|---|---|
| **Entrada** | Cubo + posición |
| **Salida (QC/productos)** | `stages/spec_aperture_qc.json` |
| **Consume aguas abajo** | D1, E1 (controles) |


## Qué hace C2 y por qué

C2 extrae el espectro del compañero por **apertura** (`box3` por defecto, `box5`) y define el **contrato de producto espectral estándar** (`wave, flux, flux_err, apcorr, npix, flags`) sobre el que se construyen **todos** los extractores (C3, C4) y D1/E1. Es infraestructura: cero ciencia nueva, la extracción ya estaba validada.

**Decisiones clave:**
- **`control = objeto`:** 33 aperturas de control se procesan **idénticas** al objeto (mismo radio, annulus de fondo, apcorr) → el σ es **empírico** (M5 STAT rojo, así que no se usa el STAT directo). [`docs/noise_model.md`](../docs/noise_model.md)
- **Corrección de apertura (growth-curve de la PSF):** box3 capta solo una fracción de la **PSF AO ancha** → una corrección grande y **cromática** (mediana ~44.7×, ~118× en el azul → ~21× en el rojo). El chequeo `v4_apcorr_range` **falla** por ese rango enorme — se marca, no se esconde.
- **Apcorr con extracción *wings-intact*** (cubo crudo + annulus), **no** el residual de 04b: 04b sobre-sustrae las alas del compañero (daría box5 < box3, no físico), así que la curva de crecimiento se mantiene auto-consistente sobre el cubo crudo.

La apertura NO es el método canónico (G1 la **rechaza** por insensible en el borde del compañero); C2 aporta el contrato de producto y el control 'A: apertura' para D1.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x01_aperture.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_aperture_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x01_aperture.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_aperture_qc.json', RUN_ID)
nb.show(qc, keys=['apertures', 'errors.mode', 'aperture_correction.median', 'v4_apcorr_range_ok', 'bad_window_channels'], title='C2')


## Los chequeos del QC, en físico

Los nombres `v2_…`/`v3_…`/`v4_…` son las verificaciones de la spec (§5) y viven así en el QC; esto es la pregunta que contesta cada una.

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v2_error_ratio_ok` | **¿La barra de error del espectro describe su dispersión real?** Compara el error propagado del cubo con el medido en controles al mismo radio (mediana de la razón dentro de [0.7, 1.4]). | Toda significancia posterior (E1, E3) queda mal escalada: el σ no es el ruido. |
| `v3_roundtrip_ok` | **¿El fichero que escribimos se relee idéntico?** Integridad del formato `SpectrumProduct` (write→read→validate). | No es física: es un producto corrupto o una versión de formato incompatible. |
| `v4_apcorr_range_ok` | **¿Sabemos cuánta luz se queda fuera de la apertura?** La corrección `apcorr(λ)` debe ser suave y estar en [1.0, ~1.6] para la caja 3×3, y el espectro corregido de 3×3 y 5×5 debe coincidir: si la curva de crecimiento de C1 es correcta, medir en caja chica o grande da lo mismo. | El flujo absoluto del compañero queda sesgado **y con dependencia en λ** (la PSF se ensancha hacia el azul). |


## Resultados que llevaron a la conclusión

Aperturas, errores empíricos, corrección de apertura y chequeos del `spec_aperture_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C2', 'stages/spec_aperture_qc.json'):
        q = nb.load_qc('stages/spec_aperture_qc.json', RUN_ID)
        err = q['errors']; ac = q['aperture_correction']; ck = q['checks']
        print('apertures:', q['apertures'], '| posiciones de:', q['positions_from'].split('/')[-1])
        print(f"errores: modo={err['mode']}, covarianza box3={err['covariance_factor_box3']:.2f}×, "
              f"stat/empírico={err['stat_vs_empirical_median_ratio']:.2f}")
        print(f"apcorr ({ac['mode']}): mediana {ac['median']:.1f}× , máx {ac['max']:.1f}× , norm r={ac['norm_radius_px']:.0f}px")
        print(f"flags: bad-window {q['flags']['bad_window_channels']}, skyline {q['flags']['skyline_channels']}, clipped {q['flags']['clipped_channels']}")
        print(f"checks: v2_error_ratio={ck['v2_error_ratio_ok']} v3_roundtrip={ck['v3_roundtrip_ok']} v4_apcorr_range={ck['v4_apcorr_range_ok']}  (v4 falla: apcorr enorme)")
        print()
        for i, s in enumerate(q['open_issues'], 1):
            print(f'  open_issue {i}: {s}')


## Plot 1 — el espectro por apertura (box3) + ruido empírico

Del producto `spec_aperture_object.fits` y los 33 controles (`spec_aperture_controls.npz`). El flujo es **muy ruidoso** (banda ±1σ empírica); el continuo suavizado **sube al rojo** (SED real de enana fría) y **no hay nada en Hα** (contexto de la no-detección). El hueco es la ventana del láser AO.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_aperture_object.fits'); d = h[1].data
    w = np.asarray(d['wave_A'], float); flux = np.asarray(d['flux'], float); h.close()
    ctrl = np.load(rd / 'stages' / 'spec_aperture_controls.npz')['control_spectra']
    sig = np.nanstd(ctrl, axis=0)
    from musepipe.spectral import median_filter_1d
    # Mediana móvil que IGNORA los NaN. Con una media y nan_to_num, los 215
    # canales sin dato (hueco del láser, bordes) entraban como CEROS y tiraban
    # la curva hacia abajo justo donde importa: ~10% en el rojo y una caída
    # falsa a cero cruzando el hueco.
    sm = median_filter_1d(flux, 41)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(w, -sig, sig, color='0.8', label=f'±1σ empírico ({ctrl.shape[0]} controles)')
    ax.plot(w, flux, lw=0.3, color='0.5', alpha=0.6)
    ax.plot(w, sm, lw=1.2, color='tab:blue', label='flujo compañero (suavizado 41ch)')
    ax.axvline(6563, color='tab:red', ls=':', label='Hα')
    ax.set_ylim(np.nanpercentile(flux, 2), np.nanpercentile(flux, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (apcorr aplicada)')
    ax.set_title('C2 · espectro por apertura box3 del compañero (errores empíricos)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c2_aperture'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'spectrum.png', dpi=110); print('figura ->', outdir / 'spectrum.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la corrección de apertura cromática

La corrección `box3 → flujo total` baja de ~118× en el azul (PSF AO peor) a ~21× en el rojo (PSF más apretada); mediana ~44.7×. Su rango enorme es lo que hace fallar `v4_apcorr_range` — es real (PSF AO ancha), no un defecto.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_aperture_object.fits'); d = h[1].data
    w = np.asarray(d['wave_A'], float); apc = np.asarray(d['apcorr'], float); h.close()
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(w, apc, lw=0.8, color='tab:purple')
    ax.axhline(np.nanmedian(apc), color='k', ls='--', lw=1, label=f'mediana {np.nanmedian(apc):.1f}×')
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('corrección de apertura ×')
    ax.set_title('C2 · corrección de apertura (box3 capta poco de la PSF AO ancha)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c2_aperture'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'apcorr.png', dpi=110); print('figura ->', outdir / 'apcorr.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'aperture'
    PRODUCT_P = 'spec_aperture_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    E_P = np.asarray(_d['flux_err_emp'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P,
               'apcorr': np.asarray(_d['apcorr'], float),
               'npix_eff': np.asarray(_d['npix_eff'], float),
               'flags': np.asarray(_d['flags'], int)}
    _h.close()
    MODO_P = nb.load_qc('stages/spec_aperture_qc.json', RUN_ID)['errors']['mode']
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · apertura box3 (C2)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c2_aperture'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / 'spectrum_paper.pdf')}))
    print('figura ->', outdir / 'spectrum_paper.pdf')
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Principio **control = objeto**: 33 controles con annulus bkg + apcorr, procesados idénticos al objeto → σ **empírico** (M5 rojo). · [`docs/noise_model.md`](../docs/noise_model.md)
- **Corrección de apertura cromática grande** (mediana 44.7×): box3 capta poco de la PSF AO ancha; `v4_apcorr_range` falla por el rango (marcado, no oculto).
- **Apcorr wings-intact** (cubo crudo + annulus), no el residual 04b, que sobre-sustrae las alas (box5<box3).
- Apertura **no canónica**: G1 la rechaza por insensible en el borde; C2 aporta el contrato de producto y el control 'A' para D1.


## Conclusión (registrada)

**C2: espectro por apertura (box3/box5) en el formato de producto estándar; errores empíricos; apcorr cromática mediana 44.7×.**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Entrada:** cubo stage02; posiciones de B3; 33 controles.
- **Espectro:** muy ruidoso, continuo real que sube al rojo (enana fría), **nada en Hα** (no-detección).
- **Errores empíricos** (M5 STAT rojo); covarianza box3 = 6.38×.
- **Apcorr:** growth-curve, 118× (azul) → 21× (rojo); `v4_apcorr_range` falla por el rango (real, PSF AO ancha); apcorr wings-intact para no sobre-sustraer alas.
- **Rol:** contrato de producto para C3/C4/D1/E1; método de apertura no canónico (psffit lo es).
